# SHAP Explainability

This notebook loads the final trained model and adds SHAP values, so every
prediction comes with a clear reason - which features pushed the score up
or down for a specific company.

In [21]:
# Loading the final features table and retraining the model.
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import xgboost as xgb

features_df = pd.read_parquet("../data/features.parquet")
features_df.shape

(2511, 12)

In [22]:
# Splitting into train and test by snapshot date, matching the modelling notebook.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [23]:
# Retraining the final model with all 9 features, for use in SHAP.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
    "director_distress_score_safe",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [24]:
# Confirming the retrained model matches the established result.
model_scores = model.predict_proba(test[feature_cols])[:, 1]

def precision_at_k(y_true, y_scores, k_percent=10):
    """Return the precision among the top k percent highest scored companies."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    return y_true.iloc[top_k_idx].mean()

precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))

np.float64(0.68)

## Feature importance and per-company explanations

In [25]:
# Explaining the model's predictions with SHAP values.
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(test[feature_cols])
shap_values.shape

(503, 9)

In [26]:
# Summarising average feature importance across all test companies.
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
importance

company_age_years                     0.612517
filing_gap_missing                    0.292701
longest_filing_gap                    0.238080
director_distress_score_safe          0.197959
days_since_last_accounts              0.160730
count_recent_resignations             0.080758
count_late_confirmation_statements    0.026763
count_new_charges                     0.024113
accounts_missing                      0.000000
dtype: float32

In [27]:
# Checking whether company age differs systematically between failed and live companies.
features_df.groupby("is_failed")["company_age_years"].describe()

,count,mean,std,min,25%,50%,75%,max
is_failed,,,,,,,,
0,1266.0,8.039640,10.970993,0.002738,1.062971,4.117728,10.784394,146.715948
1,1245.0,11.644101,13.775721,0.016427,4.065708,7.564682,13.245722,119.523614


In [28]:
# Looking at the SHAP explanation for one real company in the test set.
company_index = 0
company_number = test.iloc[company_index]["CompanyNumber"]
company_shap = shap_values[company_index]

explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
print("Company:", company_number)
print("Predicted risk score:", model_scores[company_index])
explanation

Company: 08670145
Predicted risk score: 0.60416937


company_age_years                     0.399564
days_since_last_accounts             -0.149521
filing_gap_missing                    0.097604
director_distress_score_safe         -0.063914
count_recent_resignations            -0.050053
longest_filing_gap                    0.024543
count_late_confirmation_statements    0.015675
count_new_charges                    -0.013271
accounts_missing                      0.000000
dtype: float32

In [29]:
# Turning SHAP values into a plain language explanation for one company.
def explain_prediction(company_shap: np.ndarray, feature_cols: list, top_n: int = 3) -> list[str]:
    """Return the top n features driving a prediction, as plain English sentences."""
    explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
    sentences = []
    labels = {
        "company_age_years": "the company's age",
        "days_since_last_accounts": "how recently accounts were filed",
        "count_late_confirmation_statements": "late confirmation statements",
        "count_recent_resignations": "recent director resignations",
        "count_new_charges": "new charges registered against the company",
        "longest_filing_gap": "the longest gap between filings",
        "accounts_missing": "whether an accounts due date was on record",
        "filing_gap_missing": "whether the company had enough filing history",
        "director_distress_score_safe": "whether the company's directors have a history of other company failures",
    }
    for feature, value in explanation.head(top_n).items():
        direction = "increased" if value > 0 else "decreased"
        sentences.append(f"{labels.get(feature, feature)} {direction} the risk score")
    return sentences

In [30]:
# Testing the explanation function on the same example company.
explain_prediction(company_shap, feature_cols)

["the company's age increased the risk score",
 'how recently accounts were filed decreased the risk score',
 'whether the company had enough filing history increased the risk score']

## Checking and fixing calibration

In [31]:
# Checking how well calibrated the current model's raw probabilities are.
from sklearn.calibration import calibration_curve
import numpy as np

prob_true, prob_pred = calibration_curve(test["is_failed"], model_scores, n_bins=10)
for pred, true in zip(prob_pred, prob_true):
    print(f"predicted ~{pred:.2f}  ->  actually failed {true:.2f}")

predicted ~0.08  ->  actually failed 0.02
predicted ~0.15  ->  actually failed 0.02
predicted ~0.26  ->  actually failed 0.15
predicted ~0.37  ->  actually failed 0.26
predicted ~0.45  ->  actually failed 0.36
predicted ~0.55  ->  actually failed 0.44
predicted ~0.64  ->  actually failed 0.53
predicted ~0.75  ->  actually failed 0.43
predicted ~0.85  ->  actually failed 0.57
predicted ~0.98  ->  actually failed 0.93


In [32]:
# Splitting off a calibration set from training data, never touching the test set.
calibration_size = int(len(train) * 0.2)
calibration_set = train.iloc[-calibration_size:]
train_for_model = train.iloc[:-calibration_size]

print(len(train_for_model), len(calibration_set), len(test))

1607 401 503


In [34]:
# Importing CalibratedClassifierCV, needed for the calibration step below.
from sklearn.calibration import CalibratedClassifierCV

In [35]:
# Calibrating using a held out slice of training data, never touching the test set.
calibrated_model = CalibratedClassifierCV(model, method="isotonic", cv="prefit")
calibrated_model.fit(calibration_set[feature_cols], calibration_set["is_failed"])

calibrated_test_scores = calibrated_model.predict_proba(test[feature_cols])[:, 1]

/opt/anaconda3/lib/python3.13/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [ ]:
# Honestly rechecking calibration on the untouched test set.
prob_true_cal, prob_pred_cal = calibration_curve(test["is_failed"], calibrated_test_scores, n_bins=10)
for pred, true in zip(prob_pred_cal, prob_true_cal):
    print(f"predicted ~{pred:.2f}  ->  actually failed {true:.2f}")

predicted ~0.01  ->  actually failed 0.02
predicted ~0.13  ->  actually failed 0.22
predicted ~0.25  ->  actually failed 0.48
predicted ~0.33  ->  actually failed 0.43
predicted ~0.46  ->  actually failed 0.60
predicted ~0.58  ->  actually failed 0.47
predicted ~0.77  ->  actually failed 0.52
predicted ~0.80  ->  actually failed 0.31
predicted ~1.00  ->  actually failed 0.84


In [ ]:
# Checking how many companies fall in each calibration bin, since small bins are noisy.
import pandas as pd

bins = pd.cut(calibrated_test_scores, bins=10)
pd.Series(bins).value_counts().sort_index()

(-0.001, 0.1]    125
(0.1, 0.2]       129
(0.2, 0.3]        27
(0.3, 0.4]        49
(0.4, 0.5]        10
(0.5, 0.6]        89
(0.6, 0.7]         0
(0.7, 0.8]        21
(0.8, 0.9]        16
(0.9, 1.0]        37
Name: count, dtype: int64

In [ ]:
# Rechecking calibration with fewer, wider bins so each one has enough companies to be meaningful.
prob_true_cal5, prob_pred_cal5 = calibration_curve(test["is_failed"], calibrated_test_scores, n_bins=5)
for pred, true in zip(prob_pred_cal5, prob_true_cal5):
    print(f"predicted ~{pred:.2f}  ->  actually failed {true:.2f}")

predicted ~0.07  ->  actually failed 0.13
predicted ~0.30  ->  actually failed 0.45
predicted ~0.57  ->  actually failed 0.48
predicted ~0.77  ->  actually failed 0.52
predicted ~0.94  ->  actually failed 0.68


In [ ]:
# Checking the original uncalibrated scores at the same bin width, for a fair comparison.
prob_true_orig5, prob_pred_orig5 = calibration_curve(test["is_failed"], model_scores, n_bins=5)
for pred, true in zip(prob_pred_orig5, prob_true_orig5):
    print(f"predicted ~{pred:.2f}  ->  actually failed {true:.2f}")

predicted ~0.10  ->  actually failed 0.02
predicted ~0.31  ->  actually failed 0.20
predicted ~0.51  ->  actually failed 0.41
predicted ~0.67  ->  actually failed 0.50
predicted ~0.95  ->  actually failed 0.85


In [ ]:
# Saving the calibrated model for use in the live API.
import joblib

joblib.dump(calibrated_model, "../data/calibrated_model.pkl")

['../data/calibrated_model.pkl']

In [ ]:
# Flagging which test set companies would be marked out of distribution,
# using the same logic as the live API where possible, so we can report
# both overall and in-distribution precision honestly.
def is_out_of_distribution(row) -> bool:
    return (
        row["company_age_years"] > 50
        or row["count_recent_resignations"] >= 3
    )

test = test.copy()
test["ood"] = test.apply(is_out_of_distribution, axis=1)
test["ood"].value_counts()

ood
False    493
True      10
Name: count, dtype: int64

In [ ]:
# Computing overall precision at top 10%, and precision restricted to the
# in-distribution subset, reporting both honestly rather than choosing one.
in_dist = test[~test["ood"]].reset_index(drop=True)

overall_precision = precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(calibrated_test_scores))

in_dist_scores = calibrated_model.predict_proba(in_dist[feature_cols])[:, 1]
in_dist_precision = precision_at_k(in_dist["is_failed"], pd.Series(in_dist_scores))

print("Overall precision at top 10%:", overall_precision)
print("In-distribution precision at top 10%:", in_dist_precision)
print("Out-of-distribution companies in test set:", test["ood"].sum(), "of", len(test))

Overall precision at top 10%: 0.72
In-distribution precision at top 10%: 0.7346938775510204
Out-of-distribution companies in test set: 10 of 503


## Validating the out-of-distribution detector

In [ ]:
# Defining a labelled validation set of real UK companies across categories
# the out of distribution detector should and should not flag.
validation_companies = {
    # company_number: (expected_category, should_be_flagged)
    "00445790": ("Large PLC - Tesco", True),
    "00014259": ("Bank - HSBC", True),
    "01500000": ("Large PLC - placeholder, verify", True),
}
validation_companies

{'00445790': ('Large PLC - Tesco', True),
 '00014259': ('Bank - HSBC', True),
 '01500000': ('Large PLC - placeholder, verify', True)}

In [ ]:
# Pulling a handful of real small company numbers from our own cohort,
# where we already know they are genuine small/medium private companies.
cohort = pd.read_parquet("../data/cohort.parquet")
sample_smes = cohort["CompanyNumber"].sample(5, random_state=1).tolist()
sample_smes

['06966951', '08638941', '02508134', '11566449', '04929018']

In [ ]:
# Building the full labelled validation set: real company numbers with
# their expected category and whether the out of distribution check
# should flag them.
validation_companies = {
    "00445790": ("Large PLC - Tesco", True),
    "00014259": ("Bank - HSBC", True),
    "04366849": ("Large PLC - Shell", True),
    "00048839": ("Large PLC/Bank - Barclays", True),
    "06966951": ("SME from training cohort", False),
    "08638941": ("SME from training cohort", False),
    "02508134": ("SME from training cohort", False),
    "11566449": ("SME from training cohort", False),
    "04929018": ("SME from training cohort", False),
}
len(validation_companies)

9

In [ ]:
# Testing the out of distribution detector against the live API for every
# validation company, checking whether it matches the expected label.
import requests

results = []
for number, (category, expected) in validation_companies.items():
    r = requests.get(f"http://127.0.0.1:8003/score/{number}")
    if r.status_code != 200:
        results.append({"number": number, "category": category, "expected": expected, "actual": "API_ERROR", "correct": False})
        continue
    actual = r.json().get("out_of_distribution")
    results.append({"number": number, "category": category, "expected": expected, "actual": actual, "correct": actual == expected})

results_df = pd.DataFrame(results)
results_df

,number,category,expected,actual,correct
0,00445790,Large PLC - Tesco,True,True,True
1,00014259,Bank - HSBC,True,True,True
2,04366849,Large PLC - Shell,True,API_ERROR,False
3,00048839,Large PLC/Bank - Barclays,True,API_ERROR,False
4,06966951,SME from training cohort,False,False,True
5,08638941,SME from training cohort,False,False,True
6,02508134,SME from training cohort,False,False,True
7,11566449,SME from training cohort,False,False,True
8,04929018,SME from training cohort,False,False,True


In [ ]:
# Checking exactly why Shell and Barclays failed, rather than guessing.
import requests
for number in ["04366849", "00048839"]:
    r = requests.get(f"http://127.0.0.1:8003/score/{number}")
    print(number, r.status_code, r.text[:300])

04366849 200 {"company_number":"04366849","company_name":"SHELL PLC","risk_score":0.4654,"top_reasons":[{"feature":"Company age","value":24.5,"impact":-0.5538,"direction":"decreased"},{"feature":"Longest gap between filings","value":20,"impact":0.329,"direction":"increased"},{"feature":"Recent director resignati
00048839 200 {"company_number":"00048839","company_name":"BARCLAYS PLC","risk_score":1.0,"top_reasons":[{"feature":"Company age","value":130.0,"impact":0.9599,"direction":"increased"},{"feature":"Longest gap between filings","value":15,"impact":0.363,"direction":"increased"},{"feature":"Time since accounts filed


In [37]:
# Building the full labelled validation set: real company numbers with
# their expected category and whether the out of distribution check
# should flag them.
validation_companies = {
    "00445790": ("Large PLC - Tesco", True),
    "00014259": ("Bank - HSBC", True),
    "04366849": ("Large PLC - Shell", True),
    "00048839": ("Large PLC/Bank - Barclays", True),
    "06966951": ("SME from training cohort", False),
    "08638941": ("SME from training cohort", False),
    "02508134": ("SME from training cohort", False),
    "11566449": ("SME from training cohort", False),
    "04929018": ("SME from training cohort", False),
}

In [38]:
# Re-testing the out of distribution detector against the live API, now
# that the earlier crash is resolved.
results = []
for number, (category, expected) in validation_companies.items():
    r = requests.get(f"http://127.0.0.1:8003/score/{number}")
    if r.status_code != 200:
        results.append({"number": number, "category": category, "expected": expected, "actual": "API_ERROR", "correct": False})
        continue
    actual = r.json().get("out_of_distribution")
    results.append({"number": number, "category": category, "expected": expected, "actual": actual, "correct": actual == expected})

results_df = pd.DataFrame(results)
print(results_df)
print("Accuracy:", results_df["correct"].mean())

     number                   category  expected  actual  correct
0  00445790          Large PLC - Tesco      True    True     True
1  00014259                Bank - HSBC      True    True     True
2  04366849          Large PLC - Shell      True    True     True
3  00048839  Large PLC/Bank - Barclays      True    True     True
4  06966951   SME from training cohort     False   False     True
5  08638941   SME from training cohort     False   False     True
6  02508134   SME from training cohort     False   False     True
7  11566449   SME from training cohort     False   False     True
8  04929018   SME from training cohort     False   False     True
Accuracy: 1.0


In [39]:
results = []
for number, (category, expected) in validation_companies.items():
    r = requests.get(f"http://127.0.0.1:8003/score/{number}")
    if r.status_code != 200:
        results.append({"number": number, "category": category, "expected": expected, "actual": "API_ERROR", "correct": False})
        continue
    actual = r.json().get("out_of_distribution")
    results.append({"number": number, "category": category, "expected": expected, "actual": actual, "correct": actual == expected})

results_df = pd.DataFrame(results)
print(results_df)
print("Accuracy:", results_df["correct"].mean())

     number                   category  expected  actual  correct
0  00445790          Large PLC - Tesco      True    True     True
1  00014259                Bank - HSBC      True    True     True
2  04366849          Large PLC - Shell      True    True     True
3  00048839  Large PLC/Bank - Barclays      True    True     True
4  06966951   SME from training cohort     False   False     True
5  08638941   SME from training cohort     False   False     True
6  02508134   SME from training cohort     False   False     True
7  11566449   SME from training cohort     False   False     True
8  04929018   SME from training cohort     False   False     True
Accuracy: 1.0
